In [ ]:
import json

import numpy as np

from os import path
from PIL import Image as PImage

from utils import boxpct2pix, centerpct2boxpix

OBJS_URLS = "https://raw.githubusercontent.com/acervos-digitais/herbario-data/main/json/20250705_processed.json"
IMG_URL = "https://digitais.acervos.at.eu.org/imgs/herbario/arts"
IMG_DIR = "./imgs/full"
MAX_PIXELS = 2**25
XY_OUT_DIM = (1024, 1024)
XY_CROP_MAX = 0.45
PImage.MAX_IMAGE_PIXELS = 2 * MAX_PIXELS

In [ ]:
objs_file_path = "./20250705_processed.json"

with open(objs_file_path, "r") as ifp:
  all_data = json.load(ifp)

idObjIdxs_all = [
  { "id": "Q59826388", "objIdxs": [0] },
  { "id": "Q59925035", "objIdxs": [0] },
  { "id": "Q59873488", "objIdxs": [0] },
  { "id": "Q59873115", "objIdxs": [0] },
  { "id": "Q59873308", "objIdxs": [0] },
  { "id": "Q124157343", "objIdxs": [0, 1, 3, 5] },
  { "id": "Q59504168", "objIdxs": [0] },
  { "id": "Q44083203", "objIdxs": [0] },
  { "id": "Q42571044", "objIdxs": [0, 1, 2, 3] },
  { "id": "Q9651166", "objIdxs": [0] }
]

In [ ]:
idObjIdxs_data = [x for x in idObjIdxs_all if len(x["objIdxs"]) > 0]

pix_cnts = np.zeros(XY_OUT_DIM)
pix_vals = np.zeros((*XY_OUT_DIM, 3))

for idObjIdxs in idObjIdxs_data:
  id = idObjIdxs["id"]

  img = PImage.open(path.join(IMG_DIR, f"{id}.jpg")).convert("RGB")
  iw,ih = img.size
  w_scale, h_scale = XY_OUT_DIM[0] / iw, XY_OUT_DIM[1] / ih
  crop_scale = min(w_scale, h_scale)
  siw, sih = iw * crop_scale, ih * crop_scale

  for idx in idObjIdxs["objIdxs"]:
    (x0,y0,x1,y1) = all_data[id]["objects"][idx]["box"]

    crop_w = (x1 - x0)
    crop_h = (y1 - y0)

    if crop_w > XY_CROP_MAX or crop_h > XY_CROP_MAX:
        continue

    center_x = (x0 + x1) / 2
    center_y = (y0 + y1) / 2

    dst_w = int(crop_w * siw)
    dst_h = int(crop_h * sih)

    src_x0, src_y0, src_x1, src_y1 = boxpct2pix((x0,y0,x1,y1), (iw,ih))
    dst_x0, dst_y0, dst_x1, dst_y1 = centerpct2boxpix((center_x, center_y), (dst_w, dst_h), XY_OUT_DIM)

    dst_w = dst_x1 - dst_x0
    dst_h = dst_y1 - dst_y0

    crop_vals = np.array(img.crop((src_x0, src_y0, src_x1, src_y1)).resize((dst_w, dst_h)))
    pix_vals[dst_y0:dst_y1, dst_x0:dst_x1] += crop_vals
    pix_cnts[dst_y0:dst_y1, dst_x0:dst_x1] += 1

pix_cnts_div = np.expand_dims(pix_cnts.copy(), axis=-1)
pix_cnts_div[pix_cnts == 0] = 1

pix_avg = pix_vals / pix_cnts_div
PImage.fromarray(pix_avg.astype(np.uint8))

In [ ]:
pix_cnts_div = pix_cnts.copy()
pix_cnts_div[pix_cnts == 0] = 1

all_255 = 255 * np.ones_like(pix_cnts)

alpha_out = all_255 / pix_cnts_div